# 🧠 04A — IndoBERT / Sentence-BERT Embedding (Opsi A)
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

### Pendekatan: Pre-trained Embedding (Tanpa Fine-tuning)

Model BERT digunakan **langsung** sebagai feature extractor — tidak ada proses training tambahan.
Teks diubah menjadi vektor (embedding) lalu kemiripan dihitung dengan Cosine Similarity.

```
[Teks Dosen / Skripsi]
        ↓
 ┌─────────────────────────────────────┐
 │  IndoBERT / Sentence-BERT           │
 │  (pre-trained, NO fine-tuning)      │
 │  → vektor 768 dimensi per kalimat   │
 └─────────────────────────────────────┘
        ↓
  Cosine Similarity → Top-K Rekomendasi
```

Dua model yang diuji:

| Model | Keterangan |
|-------|------------|
| `firqaaa/indo-sentence-bert-base` | Sentence-BERT versi Indonesia — **direkomendasikan** |
| `indobenchmark/indobert-base-p1` | IndoBERT — menggunakan mean pooling manual |

> ⚠️ **Aktifkan GPU dulu:** Runtime → Change runtime type → T4 GPU

---
## 🔧 LANGKAH 0 — Setup & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))
print('✅ Drive ter-mount.')

In [ ]:
!pip install -q sentence-transformers transformers
print('✅ Library siap.')

In [ ]:
import config
import pandas as pd
import numpy as np
import torch
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

# Cek GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✅ GPU aktif : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  CPU mode — proses akan lebih lambat, tapi tetap bisa berjalan.')
print(f'\n   Device digunakan: {device}')

---
## 📌 LANGKAH 1 — Load Data

In [ ]:
df_skripsi = pd.read_csv(config.FILE_SKRIPSI_CLEAN)
df_dosen   = pd.read_csv(config.FILE_DOSEN_CLEAN)

# Untuk BERT gunakan teks tanpa stemming (lebih natural)
df_skripsi['teks_bersih_bert'] = df_skripsi['teks_bersih_bert'].fillna('')
df_dosen['profil_bersih_bert'] = df_dosen['profil_bersih_bert'].fillna('')

NAMA_DOSEN = df_dosen['nama_dosen'].tolist()
PROFIL_DOSEN = df_dosen['profil_bersih_bert'].tolist()

print('✅ Data dimuat.')
print(f'   Skripsi : {len(df_skripsi)} baris')
print(f'   Dosen   : {len(df_dosen)} dosen')

---
## 📌 LANGKAH 2 — Model A: Indo Sentence-BERT

Model `firqaaa/indo-sentence-bert-base` dibangun di atas BERT arsitektur Siamese,
khusus dilatih untuk menghasilkan **sentence embedding Bahasa Indonesia** yang bisa
langsung dibandingkan dengan Cosine Similarity tanpa pooling manual.

In [ ]:
# ─── LOAD MODEL INDO SENTENCE-BERT ───────────────────────────────
print('⏳ Memuat model Indo Sentence-BERT...')
MODEL_SBERT_NAME = 'firqaaa/indo-sentence-bert-base'
model_sbert = SentenceTransformer(MODEL_SBERT_NAME, device=device)
print(f'✅ Model dimuat: {MODEL_SBERT_NAME}')
print(f'   Dimensi embedding : {model_sbert.get_sentence_embedding_dimension()}')

In [ ]:
# ─── ENCODE PROFIL DOSEN (SBERT) ─────────────────────────────────
# ⏳ Estimasi: ~1-2 menit dengan GPU
print('⏳ Encoding profil dosen dengan Indo Sentence-BERT...')

embeddings_dosen_sbert = model_sbert.encode(
    PROFIL_DOSEN,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2 normalisasi → cosine sim = dot product
)

print(f'\n✅ Shape embedding dosen (SBERT): {embeddings_dosen_sbert.shape}')
print(f'   {len(NAMA_DOSEN)} dosen × {embeddings_dosen_sbert.shape[1]} dimensi')

In [ ]:
# ─── SIMPAN EMBEDDING SBERT ──────────────────────────────────────
sbert_path = os.path.join(config.MODELS_DIR, 'embeddings_dosen_sbert.npy')
np.save(sbert_path, embeddings_dosen_sbert)
print(f'💾 Embedding SBERT dosen tersimpan: {sbert_path}')

---
## 📌 LANGKAH 3 — Model B: IndoBERT (Mean Pooling)

IndoBERT (`indobenchmark/indobert-base-p1`) adalah model BERT standar.
Karena bukan Sentence-BERT, diperlukan **mean pooling** pada hidden states
untuk mendapatkan satu vektor per kalimat.

In [ ]:
# ─── LOAD MODEL INDOBERT ─────────────────────────────────────────
print('⏳ Memuat model IndoBERT...')
MODEL_INDOBERT_NAME = config.MODEL_INDOBERT
tokenizer_ib = AutoTokenizer.from_pretrained(MODEL_INDOBERT_NAME)
model_ib     = AutoModel.from_pretrained(MODEL_INDOBERT_NAME).to(device)
model_ib.eval()
print(f'✅ Model dimuat: {MODEL_INDOBERT_NAME}')

In [ ]:
# ─── FUNGSI MEAN POOLING ─────────────────────────────────────────
def mean_pooling(model_output, attention_mask):
    """Rata-rata token embeddings dengan mempertimbangkan attention mask (abaikan padding)."""
    token_embeddings = model_output.last_hidden_state
    mask_expanded    = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask_expanded, 1) / torch.clamp(mask_expanded.sum(1), min=1e-9)


def encode_indobert(texts, batch_size=8, max_length=128):
    """Encode list teks menjadi embedding menggunakan IndoBERT + mean pooling."""
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding IndoBERT'):
        batch = texts[i : i + batch_size]
        encoded = tokenizer_ib(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            output = model_ib(**encoded)

        embeddings = mean_pooling(output, encoded['attention_mask'])
        # L2 normalisasi
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

print('✅ Fungsi encode_indobert() siap.')

In [ ]:
# ─── ENCODE PROFIL DOSEN (INDOBERT) ──────────────────────────────
# ⏳ Estimasi: ~2-5 menit dengan GPU
print('⏳ Encoding profil dosen dengan IndoBERT...')
embeddings_dosen_ib = encode_indobert(PROFIL_DOSEN, batch_size=8)

print(f'\n✅ Shape embedding dosen (IndoBERT): {embeddings_dosen_ib.shape}')

# Simpan
ib_path = config.FILE_EMBEDDINGS_DOSEN
np.save(ib_path, embeddings_dosen_ib)
print(f'💾 Embedding IndoBERT dosen tersimpan: {ib_path}')

---
## 📌 LANGKAH 4 — Fungsi Rekomendasi (Kedua Model)

In [ ]:
def rekomendasikan_bert(judul_query, model_type='sbert', top_k=5, verbose=True):
    """
    Rekomendasi dosen menggunakan BERT embedding.

    Parameters:
        judul_query : str        — judul atau topik skripsi
        model_type  : str        — 'sbert' atau 'indobert'
        top_k       : int        — jumlah rekomendasi
        verbose     : bool
    """
    # Encode query
    if model_type == 'sbert':
        query_emb     = model_sbert.encode([judul_query], normalize_embeddings=True)
        dosen_emb     = embeddings_dosen_sbert
        label_model   = 'Indo Sentence-BERT'
    elif model_type == 'indobert':
        query_emb     = encode_indobert([judul_query], batch_size=1)
        dosen_emb     = embeddings_dosen_ib
        label_model   = 'IndoBERT (Mean Pooling)'
    else:
        raise ValueError("model_type harus 'sbert' atau 'indobert'")

    # Cosine similarity (karena sudah L2-normalized, bisa pakai dot product)
    scores  = cosine_similarity(query_emb, dosen_emb).flatten()
    ranking = np.argsort(scores)[::-1]
    hasil   = [(NAMA_DOSEN[i], round(float(scores[i]), 4)) for i in ranking[:top_k]]

    if verbose:
        print(f'🔍 Query  : "{judul_query}"')
        print(f'🤖 Model  : {label_model}')
        print(f'📋 Top-{top_k} Rekomendasi:')
        print('-' * 65)
        for rank, (nama, score) in enumerate(hasil, 1):
            bar = '█' * int(score * 25)
            print(f'  {rank}. {nama[:40]:<42} {score:.4f} {bar}')
        print()

    return hasil

print('✅ Fungsi rekomendasikan_bert() siap.')

In [ ]:
# ─── UJI COBA REKOMENDASI ─────────────────────────────────────────
contoh_judul = [
    'Implementasi Deep Learning untuk Deteksi Penyakit Tanaman pada Citra Digital',
    'Pembangunan Sistem Informasi Manajemen Arsip Berbasis Web',
    'Klasifikasi Teks Bahasa Indonesia Menggunakan Transformer',
]

for judul in contoh_judul:
    print('=' * 65)
    rekomendasikan_bert(judul, model_type='sbert',     top_k=3)
    rekomendasikan_bert(judul, model_type='indobert',  top_k=3)
    print()

---
## 📌 LANGKAH 5 — Evaluasi Top-K Accuracy (Kedua Model)

In [ ]:
# ─── SIAPKAN DATA TEST ───────────────────────────────────────────
dosen_valid = set(NAMA_DOSEN)
df_eval = df_skripsi[df_skripsi['pembimbing'].isin(dosen_valid)].copy().reset_index(drop=True)
_, df_test = train_test_split(df_eval, test_size=0.2, random_state=42, stratify=df_eval['pembimbing'])
print(f'Data test: {len(df_test)} skripsi')

In [ ]:
# ─── ENCODE SEMUA QUERY TEST SEKALIGUS (lebih efisien) ───────────
print('⏳ Encoding semua query test...')

query_texts = df_test['teks_bersih_bert'].tolist()

# SBERT
q_emb_sbert = model_sbert.encode(query_texts, batch_size=32,
                                  show_progress_bar=True, normalize_embeddings=True)
# IndoBERT
q_emb_ib    = encode_indobert(query_texts, batch_size=16)

print(f'\n✅ Shape query embedding (SBERT)    : {q_emb_sbert.shape}')
print(f'   Shape query embedding (IndoBERT) : {q_emb_ib.shape}')

In [ ]:
# ─── HITUNG EVALUASI ──────────────────────────────────────────────
def evaluasi_embedding(q_embeddings, d_embeddings, df_test, nama_dosen, top_k_list=[1,3,5]):
    """Evaluasi Top-K Accuracy menggunakan pre-computed embeddings."""
    # Hitung semua similarity sekaligus (matrix operation — sangat cepat)
    sim_matrix = cosine_similarity(q_embeddings, d_embeddings)  # shape: (n_test, n_dosen)
    results    = {k: 0 for k in top_k_list}
    errors     = []

    for idx, (_, row) in enumerate(df_test.iterrows()):
        gt      = row['pembimbing']
        scores  = sim_matrix[idx]
        ranking = np.argsort(scores)[::-1]
        ordered = [nama_dosen[i] for i in ranking]

        for k in top_k_list:
            if gt in ordered[:k]:
                results[k] += 1
            elif k == 1:
                errors.append({'judul': row['judul'], 'gt': gt, 'pred': ordered[0]})

    n = len(df_test)
    return {k: round(v/n*100, 2) for k, v in results.items()}, errors

print('⏳ Evaluasi SBERT...')
acc_sbert, err_sbert = evaluasi_embedding(q_emb_sbert, embeddings_dosen_sbert, df_test, NAMA_DOSEN)

print('⏳ Evaluasi IndoBERT...')
acc_ib, err_ib = evaluasi_embedding(q_emb_ib, embeddings_dosen_ib, df_test, NAMA_DOSEN)

print('\n' + '='*55)
print('📊 HASIL EVALUASI — BERT Embedding (Opsi A)')
print('='*55)
print(f'{"Model":<30} {"Top-1":>8} {"Top-3":>8} {"Top-5":>8}')
print('-'*55)
print(f'{"Indo Sentence-BERT":<30} {acc_sbert[1]:>7.1f}% {acc_sbert[3]:>7.1f}% {acc_sbert[5]:>7.1f}%')
print(f'{"IndoBERT (Mean Pooling)":<30} {acc_ib[1]:>7.1f}% {acc_ib[3]:>7.1f}% {acc_ib[5]:>7.1f}%')
print('='*55)

In [ ]:
# ─── VISUALISASI PERBANDINGAN ─────────────────────────────────────
import pandas as pd
existing_eval = pd.read_csv(config.FILE_EVALUATION)

# Tambahkan hasil BERT ke tabel evaluasi
new_rows = pd.DataFrame([
    {'metode': 'Indo Sentence-BERT (Embedding)',
     'top1_accuracy': acc_sbert[1], 'top3_accuracy': acc_sbert[3], 'top5_accuracy': acc_sbert[5],
     'n_test': len(df_test)},
    {'metode': 'IndoBERT Mean Pooling (Embedding)',
     'top1_accuracy': acc_ib[1], 'top3_accuracy': acc_ib[3], 'top5_accuracy': acc_ib[5],
     'n_test': len(df_test)},
])
df_eval_all = pd.concat([existing_eval, new_rows], ignore_index=True)
df_eval_all.to_csv(config.FILE_EVALUATION, index=False)

print('📊 Tabel evaluasi lengkap:')
print(df_eval_all[['metode','top1_accuracy','top3_accuracy','top5_accuracy']].to_string(index=False))

# Plot perbandingan
fig, ax = plt.subplots(figsize=(12, 5))
x    = np.arange(3)
w    = 0.25
ks   = ['Top-1', 'Top-3', 'Top-5']
cols = ['top1_accuracy','top3_accuracy','top5_accuracy']
colors = ['#1565C0', '#2E7D32', '#E65100']

for i, (_, row) in enumerate(df_eval_all.iterrows()):
    vals = [row[c] for c in cols]
    bars = ax.bar(x + i*w, vals, width=w, label=row['metode'], color=colors[i % len(colors)], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.1f}%',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + w)
ax.set_xticklabels(ks)
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Perbandingan Top-K Accuracy: TF-IDF vs BERT Embedding', fontweight='bold', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.axhline(100, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'bert_embedding_eval.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 📌 LANGKAH 6 — Demo Interaktif

In [ ]:
# ─── DEMO: MASUKKAN JUDUL SKRIPSI SENDIRI ────────────────────────
JUDUL_QUERY = "Sistem Deteksi Hoaks Berbasis Natural Language Processing Menggunakan IndoBERT"

print('=' * 65)
print('🤖 SBERT:')
rekomendasikan_bert(JUDUL_QUERY, model_type='sbert',    top_k=5)
print('🤖 IndoBERT:')
rekomendasikan_bert(JUDUL_QUERY, model_type='indobert', top_k=5)
print('=' * 65)

---
## ✅ Selesai — Ringkasan Notebook 04A (Opsi A)

| Output | Lokasi |
|--------|--------|
| Embedding SBERT dosen | `models/embeddings_dosen_sbert.npy` |
| Embedding IndoBERT dosen | `models/embeddings_dosen.npy` |
| Hasil evaluasi (update) | `results/evaluation.csv` |
| Grafik perbandingan | `results/bert_embedding_eval.png` |

### 🗺️ Langkah Berikutnya:
> Jika menggunakan Opsi B (fine-tuning SetFit), lanjut ke **`04B_setfit.ipynb`**
> Jika hanya Opsi A, lanjut ke **`05_evaluasi.ipynb`**